In [9]:
from pathlib import Path
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
import numpy as np
import pickle
import random
import SimpleITK as sitk
import cv2
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
import numpy as np
from torchvision import transforms
import mysql.connector
import os
from PIL import ImageOps
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import vtk
from scipy.spatial.transform import Rotation as R
from scipy.interpolate import RegularGridInterpolator
import matplotlib.pyplot as plt


In [10]:
res_size = (96, 48, 128)

base_folder = Path(r"/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp")
skull_file = base_folder / "combined_skull_analysis.xlsx"

images_folder = r"/home/varhai/PRIMUS/var_vestlontest/export/images/"
conn = mysql.connector.connect(option_files=os.path.expanduser('~/.my.cnf')) 
sql_query = """
SELECT o.SPECIMEN_ID,s.STRAIN,s.GENDER, m.COHORT_ID, e.DATE,o.VALUE_CATEGORY as SkullShape, LEN.VALUE_DATA_POINT as CraniumLength,COM.VALUE_TEXT as Comment
FROM OBSERVATION as o
    INNER JOIN MOUSE as m ON o.SPECIMEN_ID = m.ID
    INNER JOIN SPECIMEN as s ON m.ID = s.ID
    INNER JOIN EXPERIMENT as e ON e.ID = o.EXPERIMENT_ID
    INNER JOIN PARAMETER as p ON o.PARAMETER_STABLE_ID = p.PARAMETER_STABLE_ID
    LEFT JOIN (
        SELECT SPECIMEN_ID, VALUE_TEXT
        FROM OBSERVATION
        WHERE PARAMETER_STABLE_ID = 'CCP_XRY_032_001'
    ) AS COM ON o.SPECIMEN_ID = COM.SPECIMEN_ID

    LEFT JOIN (
        SELECT SPECIMEN_ID, VALUE_DATA_POINT
        FROM OBSERVATION
        WHERE PARAMETER_STABLE_ID = 'CCP_XRY_893_001'
    ) AS LEN ON o.SPECIMEN_ID = LEN.SPECIMEN_ID
WHERE
    m.COHORT_ID LIKE 'c%'
    AND o.PARAMETER_STABLE_ID = 'CCP_XRY_001_001';
"""
# Reading data into a Pandas DataFrame
db_df_test = pd.read_sql(sql_query, conn)
skull_df = pd.read_excel(skull_file)

<ipython-input-10-71c5e1e08e1a>:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_df_test = pd.read_sql(sql_query, conn)


In [11]:
# match sutures with database
# Defining your SQL query


sql_query = """
SELECT SPECIMEN_ID, VALUE_IMAGE_URI FROM `OBSERVATION` WHERE `PARAMETER_STABLE_ID` = 'CCP_XRY_051_001' AND `INCREMENT_VALUE` = '001'
 and VALUE_IMAGE_URI is not null 
"""
# Reading data into a Pandas DataFrame
db_df = pd.read_sql(sql_query, conn)

# Closing the connection
conn.close()
display(db_df)

<ipython-input-11-d2dfdf71a5cf>:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_df = pd.read_sql(sql_query, conn)


,SPECIMEN_ID,VALUE_IMAGE_URI
0,064-EPJ77,2018-07-11/nub1_f_83z-epj77_cr1.bmp
1,064-EPJ81,2018-07-11/nub1_m_064-epj81_cr1.bmp
2,064-EPJ82,2018-07-11/nub1_m_064-epj82_cr1.bmp
3,064-EPJ85,2018-07-11/nub1_f_83z-epj85_cr1.bmp
4,064-EPJ86,2018-07-11/nub1_f_83z-epj86_cr1.bmp
...,...,...
10627,E83-F2740,2020-08-24/fam161b_m_e83-f2740_cr1.bmp
10628,E83-F2743,2020-08-24/fam161b_f_e83-f2743_cr1.bmp
10629,E83-F2749,2021-01-22/fam161b_f_e83-f2749_cr1.bmp
10630,E83-F2750,2021-01-22/fam161b_f_e83-f2750_cr1.bmp


# Filter subjects with skull data which have sutures

In [12]:
# merge the dataframes
skull_df["Specimen_ID"] = skull_df["Unnamed: 0"].apply(lambda x: x.split("_")[-1])
skull_df = skull_df.merge(db_df, left_on="Specimen_ID", right_on="SPECIMEN_ID")


skull_df = skull_df[skull_df['proc_dist']<3000]


In [13]:
# get 2 classes for sutures (equal number of sutures and no sutures)

#no_sutures_df = no_sutures_df.sample(n=sutures_df.shape[0], random_state=42)

In [14]:
# generate train and test split take equal number of sutures and no sutures for each
train_df = skull_df.sample(frac=0.8, random_state=42)
test_df = skull_df.drop(train_df.index)
# randomize the dataset
train_df = train_df.sample(frac=1, random_state=42)
test_df = test_df.sample(frac=1, random_state=42)

In [7]:
test_df.describe()

,DATE,CraniumLength,proc_dist,area99perc,s_x,s_y,s_z
count,789,770.000000,789.000000,789.000000,789.000000,789.000000,789.000000
mean,2022-06-14 07:43:34.448669184,26.361991,742.647587,0.059963,1.028549,0.992273,0.958969
min,2020-09-28 00:00:00,21.550000,218.461700,0.000000,0.879162,0.831809,0.621642
25%,2021-09-20 00:00:00,26.020000,463.334501,0.000884,1.006299,0.979774,0.952195
50%,2022-06-13 00:00:00,26.400000,577.782632,0.005803,1.022000,0.993386,0.965572
75%,2023-02-27 00:00:00,26.780000,773.474272,0.021790,1.038710,1.006319,0.979331
max,2023-11-20 00:00:00,28.390000,2989.457342,0.999693,2.120344,1.578354,1.383324
std,NaN,0.677592,506.679115,0.187133,0.054225,0.032137,0.045388


In [39]:
from typing import Tuple
import numpy as np
import nibabel as nib
import vtk
from scipy.spatial.transform import Rotation as R
import cv2
from tqdm import tqdm
import os
from PIL import ImageOps
import torch
import torchvision.transforms as transforms

from sys import getsizeof

def flip_image(im_file_in: str, im_file_out: str, axis:Tuple[int,int,int]):
    # Load your image (replace 'image_path' with the path to your image)
    image = sitk.ReadImage(im_file_in)
    # Flip the image along a specific axis (e.g., along the x-axis)
    # Flip the image along the x-axis
    #flipped_image = sitk.Flip(image, [True, False, False])  # [X, Y, Z] axis flipping
    # Specify the axis along which you want to mirror (0 for x-axis, 1 for y-axis, 2 for z-axis)
    axis_to_mirror = 0  # For example, mirror along the x-axis
    image_array = sitk.GetArrayFromImage(image)
    # Flip the image along the specified axis
    mirrored_image_array = np.flip(image_array, axis=axis#'(1,2)
    )

    # Create a SimpleITK image from the mirrored NumPy array
    flipped_image = sitk.GetImageFromArray(mirrored_image_array)

    # Set the origin, spacing, and direction cosine matrix if needed
    flipped_image.SetOrigin(image.GetOrigin())
    flipped_image.SetSpacing(image.GetSpacing())
    flipped_image.SetDirection(image.GetDirection())

    # Apply the flipping transformation to the image

    sitk.WriteImage(flipped_image, im_file_out)


def get_head_file_n_mesh(subj_id, cohort_id):
    """
    Get the head file and mesh file for a given subject and cohort
    :param subj_id: 
    :param cohort_id: 
    :return: 
    """
    folders = [x for x in base_folder.iterdir() if cohort_id in x.name and (x/"skull_shape_workdir").exists()]
    for fold in folders:
        files = list((fold/"skull_shape_workdir").iterdir())
        for file in files:
            if str(subj_id) in file.name:
                if (file/"head_affine_registered.obj").exists() and (file/"oriented_image.nii.gz").exists()\
                        and (file/"affine_head.mat").exists():
                    filenames_ = ["coreg_1.mat", "coreg_2.mat", "coreg_3.mat","coreg_or.mat"]
                    # check which files not exist from filenames_
                    exist_files = {x: (file/x).exists() for x in filenames_}
                    # check that exist exactly 3 files
                    if sum(exist_files.values()) == 3:
                        # check which file not exist
                        not_exist_file = [x for x in filenames_ if not exist_files[x]][0]
                        if (file/"ct_image_.nii.gz").exists():
                            ctim = str(file/"ct_image_.nii.gz")
                        elif (file/"ct_image.nii.gz").exists():
                            ctim = str(file/"ct_image.nii.gz")
                        #flip_image(ctim, str(file / "preregister_image1.nii.gz"),axis=(1,2))
                        #flip_image(ctim, str(file / "preregister_image2.nii.gz"),axis=(0,2))
                        #flip_image(ctim, str(file / "preregister_image3.nii.gz"),axis=(0,1))
                        
                        if not_exist_file == "coreg_1.mat":
                            flip_image(ctim, str(file / "preregister_image.nii.gz"),axis=(1,2))
                        elif not_exist_file == "coreg_2.mat":
                            flip_image(ctim, str(file / "preregister_image.nii.gz"),axis=(0,2))
                        elif not_exist_file == "coreg_3.mat":
                            flip_image(ctim, str(file / "preregister_image.nii.gz"),axis=(0,1))
                        elif not_exist_file == "coreg_or.mat":
                            # copy ct_image to preregister_image
                            os.system(f"cp {ctim} {str(file/'preregister_image.nii.gz')}")
                        return file/"head_affine_registered.obj", file/"preregister_image.nii.gz", file/"affine_head.mat"
                            
                    else:
                        return None, None, None

    return None, None, None


def get_points_from_mesh(mesh_file):
    pt1_id = [9915]
    pt2_id = [9304]
    pt3_id = [17278, 16945]

    reader = vtk.vtkOBJReader()
    reader.SetFileName(str(mesh_file))
    reader.Update()
    polydata = reader.GetOutput()

    points = polydata.GetPoints()
    pts1 = [points.GetPoint(pt1_id[0])]
    
    #print("pts1 orig", pts1)
    pts2 = [points.GetPoint(pt2_id[0])]
    #print("pts2 orig", pts2)
    pts3 = [points.GetPoint(pt3_id[0]), points.GetPoint(pt3_id[1])]
    return np.mean(pts1, axis=0), np.array(pts2[0]), np.mean(pts3, axis=0)


def world_to_voxel(world_coords, affine):
    """Convert world coordinates to voxel coordinates."""
    inv_affine = np.linalg.inv(affine)
    voxel_coords = nib.affines.apply_affine(inv_affine, world_coords)
    return voxel_coords


def read_3d_image(image_file_name, affine_file_name, points_world):
    """
    Read a 3D image and transform using the affine matrix and from world to voxel coordinates
    :param image_file_name: 
    :param affine_file_name: 
    :param points_world: 
    :return: 
    """
    #print(image_file_name)
    head_img = nib.load(image_file_name)
    head_data = head_img.get_fdata()
    head_affine = head_img.affine
    #print("pw_orig", points_world)
    #read affine matrix frin reg_aladin output
    affine = np.loadtxt(affine_file_name)
    #print("orig_2_reg", affine)
    # transform points from coreg image to image unregistred
    points_in_image_world = world_to_voxel(points_world, np.linalg.inv(affine))
    
    print(points_in_image_world)
    # get the points in voxel images 
    points_in_image_voxel = world_to_voxel(points_in_image_world, head_affine)
    #print(points_in_image_voxel)
    # get the image 3d
    
    # return the image data and the points in voxel coordinates
    return head_data, points_in_image_voxel
    
def get_parallel_plane(head_data, grid_points, 
                       head_affine, normal_vector, d, num_points=400):
    # Interpolate the image data points shifted by the normal vector and d
    dims = head_data.shape
    x = np.arange(dims[0])
    y = np.arange(dims[1])
    z = np.arange(dims[2])
    interpolator = RegularGridInterpolator((x, y, z), head_data, method='linear', bounds_error=False, fill_value=0)
    
    # shift the grid points by the normal vector and d
    grid_points_shifted = grid_points + d * normal_vector
    
    # Convert the grid points back to voxel coordinates
    grid_points_voxel = nib.affines.apply_affine(np.linalg.inv(head_affine), grid_points_shifted)
    
    return interpolator(grid_points_voxel).reshape(num_points, num_points)

def read_subject(subj_id, cohort_id):
    mesh_file, head_file, affine_file  = get_head_file_n_mesh(subj_id, cohort_id)
    if mesh_file is None:
        return None,None

    points_world = get_points_from_mesh(mesh_file)
    image_data, points_result = read_3d_image(head_file, affine_file, points_world)
    return image_data, points_result



def read_subjs(df, process=lambda x: x, is_process=False, folder=None):
    results = []
    for i in tqdm(range(df.shape[0])):
        subj = dict(df.iloc[i])
        #print(i)
        image, points = read_subject(subj['Specimen_ID'], subj['COHORT_ID'])
        #print(torch.from_numpy(image).unsqueeze(0).unsqueeze(0).shape)

        #print(image.shape)
        
        #print(image.nbytes)
        if image is None:
            continue
        shape = image.shape
        image = F.interpolate(torch.from_numpy(image).unsqueeze(0).unsqueeze(0), size=(128, 128, 128), mode='trilinear', align_corners=False)
        image = image.squeeze(0).squeeze(0).numpy()
        res = {"points": points, "image": image, "shape" : shape}
        results.append(res)
        if is_process:
            pass
        #break
    return results


# Read training and test data
train_data = read_subjs(train_df, is_process=False, folder=r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_set_proj')
test_data = read_subjs(test_df, is_process=False, folder=r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_set_proj')


  0%|          | 0/3155 [00:00<?, ?it/s]

pts1 orig [(-4.686110019683838, -3.196491241455078, 18.314809799194336), (-4.527708053588867, -3.4346041679382324, 18.25750732421875)]
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20210802c200_Agpat5 & Chuk & Fam81a & Meis3 & MMP20 & wt/skull_shape_workdir/wt_M_82Z-07463/oriented_image.nii.gz


  0%|          | 1/3155 [00:04<3:41:47,  4.22s/it]

pw_orig (array([-4.60690904, -3.3155477 , 18.28615856]), array([ 6.48039365, -3.51486242, 17.47999001]), array([ 0.54361714, -4.59637499, 25.21542263]))
[[20.28308528 12.97089763 26.68359528]
 [31.08939269 11.51717877 26.00716085]
 [25.21324788 11.75832741 33.72868508]]
[[-392.86225789 -251.23279121  516.83347705]
 [-602.16918878 -223.07576944  503.7316459 ]
 [-488.35437767 -227.74656778  653.28953623]]
pts1 orig [(-4.741229057312012, -3.3283684253692627, 18.315532684326172), (-4.572409629821777, -3.5988001823425293, 18.296733856201172)]
/home/varhai/PRIMUS/data/83_IMPC_DATA/Bioimaging/SCREENad.Morpho&BodyComp/Screen20230619c293_Dis3l & Nkiras1 & psKlcs & wt/skull_shape_workdir/psKlcs_M_E64-CE170/oriented_image.nii.gz


  0%|          | 1/3155 [00:05<4:37:47,  5.28s/it]


KeyboardInterrupt: 

In [ ]:
# reload the data after manual removal of heads

def read_from_files(folder):
    results = []
    # Iterate over each file in the directory
    for filename in tqdm(os.listdir(folder)):
        # Construct the full image path
        image_path = os.path.join(folder, filename)
        
        # Extract class information from the filename
        clas_res = True if "True" in filename else False
        
        # Read the image in grayscale
        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        # Create the result dictionary
        res = {"class": clas_res, "image": image}
        
        # Append the result to the results list
        results.append(res)
    
    return results



train_data = read_from_files(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_set_sutures')
test_data = read_from_files(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_set_sutures')



In [ ]:
# # save train/ test data to a file
# if train_data is not None:
#     with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_data_sutures_proj.pkl', 'wb') as f:
#         pickle.dump(train_data, f)
# else:
#     with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/train_data_sutures_proj.pkl', 'rb') as f:
#         train_data = pickle.load(f)
# if test_data is not None:
#     with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures_proj.pkl', 'wb') as f:
#         pickle.dump(test_data, f)
# else:
#     with open(r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures_proj.pkl', 'rb') as f:
#         test_data = pickle.load(f)




In [29]:
# count how many sutures and no sutures per train and test data

print("Train", sum([1 for x in train_data if x['class'] == True]))
print("Test", sum([1 for x in test_data if x['class'] == True]))


KeyError: 'class'

In [31]:
class MouseSuturesDataset(Dataset):
    def __init__(self, data, augment=False):
        self.data = data
        self.augment = augment

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx]['image']
        points = self.data[idx]['points']
        shape = self.data[idx]['shape']
        if self.augment and random.random() > 0.5:
            image, points = self._augment_image(image, points)
            
        # Normalize the points to be within [0, 1] range (points are in voxel coordinates so we divide by the image size)
        points = points / shape
        
        # Convert the image to a PyTorch tensor
        image = torch.tensor(image, dtype=torch.float32)
        


        return image, points


    def _augment_image(self, image, points):
        # Randomly choose which augmentation to apply
        if random.random() > 0.2:
            image, points = self._random_shift(image, points)
#         if random.random() > 0.2:
#             image, points = self._add_gaussian_noise(image, points)
        if random.random() > 0.2:
            image, points = self._flip_image(image, points)
            
        return image, points

    def _random_shift(self, image, points, max_shift=5):
        # 3d shift of the image and points
        shift_x = random.randint(-max_shift, max_shift)
        shift_y = random.randint(-max_shift, max_shift)
        shift_z = random.randint(-max_shift, max_shift)
        image = np.roll(image, shift_x, axis=0)
        image = np.roll(image, shift_y, axis=1)
        image = np.roll(image, shift_z, axis=2)
        points = points + np.array([shift_x, shift_y, shift_z])
        return image, points

    def _flip_image(self, image, points):
        # Flip the image and points along the random axis
        axis = random.choice([0, 1, 2])
        image = np.flip(image, axis=axis)
        points[axis] = image.shape[axis] - points[axis]
        return image.copy(), points

    def _add_gaussian_noise(self, image, mean=0, std=0.1):
        noise = torch.randn(image.size()) * std + mean
        image = image + noise
        image = torch.clamp(image, 0, 1)  # Ensure values are within [0, 1]
        return image,points

In [ ]:

save_folder = r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures'#,process_image

In [ ]:
test_df.to_csv("/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/test_data_sutures/test_names.csv")

In [ ]:
sutures_df.describe()

In [ ]:
import torchvision

In [ ]:
torchvision.models.googlenet()

In [32]:


train_dataset = MouseSuturesDataset(train_data, True)
test_dataset = MouseSuturesDataset(test_data, False)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

# 
# # Replace the final layer with a custom one
# num_ftrs = resnet50.fc.in_features
# resnet50.fc = nn.Linear(num_ftrs, 1).to(device)  # Assuming you're doing a regression task

In [34]:
from matplotlib.pyplot import imsave

# define Neural Network

class FCN(nn.Module):
    def __init__(self):
        super(FCN, self).__init__()
        self.conv1 = nn.Conv3d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool3d(2)
        self.conv2 = nn.Conv3d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool3d(2)
        self.conv3 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool3d(2)
        self.global_pool = nn.AdaptiveAvgPool3d(1)
        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x


num_epochs = 500


best_val_loss = 1000


criterion = nn.L1Loss()
#vgg = resnet50
vgg = FCN().to(device)
optimizer = torch.optim.Adam(vgg.parameters(), lr=1e-5)

early_stopping_patience = 15

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
epoch_losses = []
val_losses = []
for epoch in range(500):
    vgg.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader):
        inputs = inputs.unsqueeze(0)
        print(outputs)
        inputs = F.interpolate(inputs, size=(128, 128, 128), mode='trilinear', align_corners=False)
        inputs, labels = (inputs.to(device), labels.to(device))

        
        optimizer.zero_grad()
        outputs = vgg(inputs)
        outputs = torch.sigmoid(outputs)
        #print(outputs)
        #print(outputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_loader.dataset)
    vgg.eval()
    print("val")
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.unsqueeze(0)
            inputs = F.interpolate(inputs, size=(128, 128, 128), mode='trilinear', align_corners=False)
            inputs, labels = (inputs.to(device), labels.to(device))
            outputs = vgg(inputs)
            outputs = torch.sigmoid(outputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() #* inputs.size(0)
    
    val_loss = val_loss / len(test_loader.dataset)
    val_losses.append(val_loss)
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {epoch_loss:.4f}, Validation Loss: {val_loss:.4f}')
    epoch_losses.append(epoch_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(vgg.state_dict(), 'best_model_sutures_gen.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print("Early stopping triggered")
            break  

  0%|          | 1/3141 [00:00<23:48,  2.20it/s]

tensor([[0.5000, 0.5000, 0.7311]], device='cuda:0')
tensor([[0.6540, 0.6004, 0.6148]], device='cuda:0', grad_fn=<SigmoidBackward0>)


  0%|          | 3/3141 [00:00<11:34,  4.52it/s]

tensor([[0.6414, 0.6069, 0.6172]], device='cuda:0', grad_fn=<SigmoidBackward0>)
tensor([[0.6427, 0.6046, 0.6175]], device='cuda:0', grad_fn=<SigmoidBackward0>)


  0%|          | 5/3141 [00:01<09:25,  5.55it/s]

...

  4%|▍         | 121/3141 [00:19<08:06,  6.21it/s]

tensor([[0.5522, 0.5298, 0.7052]], device='cuda:0', grad_fn=<SigmoidBackward0>)
tensor([[0.5471, 0.5252, 0.7094]], device='cuda:0', grad_fn=<SigmoidBackward0>)


  4%|▍         | 122/3141 [00:20<08:16,  6.07it/s]

tensor([[0.5504, 0.5286, 0.7063]], device='cuda:0', grad_fn=<SigmoidBackward0>)


KeyboardInterrupt: 

In [ ]:
true_positives = 0
true_negatives = 0
false_positives = 0
false_negatives = 0
torch.cuda.empty_cache()
vgg.eval()
#vgg.to('cpu')
# load the best model
vgg.load_state_dict(torch.load('best_model_sutures_gen.pth'))
torch.cuda.empty_cache()
#vgg = vgg.to('cpu')
result_computed = []
folder_save = r'/home/varhai/PRIMUS/data/83_BIOINFORMATICS/Igor/workdir/false_detections_sutures'
i=0
with torch.no_grad():
    for inputs, labels in tqdm(test_loader):
        orig_inp = inputs
        #inputs, labels = transform(inputs.squeeze(0).float()).to('cpu'), labels.float().to('cpu')
        inputs, labels = (inputs.squeeze(0).squeeze(0).float().to(device),#transform(inputs.squeeze(0).float()).to(device), 
                          labels.float().to(device))
    
        outputs = vgg(inputs.unsqueeze(0))
        outputs = torch.sigmoid(outputs)
        #print(outputs)
        # save outputs and label to a list
        result_computed.append((outputs, labels))
        predicted_class = outputs > 0.5 # threshold at 0.5
        true_class = labels
        if true_class == 1 and predicted_class == 1:
            true_positives += 1
        elif true_class == 0 and predicted_class == 0:
            true_negatives += 1
        elif true_class == 0 and predicted_class == 1:
            false_positives += 1
            # save fp image to a folder
            #print(orig_inp.shape)
#             imsave(os.path.join(folder_save, f'fp_{i}.png'), inputs.squeeze(0)
#                    .cpu().numpy()) 
        elif true_class == 1 and predicted_class == 0:
            false_negatives += 1
            #imsave(os.path.join(folder_save, f'fn_{i}.png'), inputs.squeeze(0).cpu().numpy()) 
        torch.cuda.empty_cache()
        i+=1

accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
f1_score = 2 * precision * recall / (precision + recall)
print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}')
        #print(outputs.argmax(0))

In [ ]:
print(true_positives, true_negatives, false_positives , false_negatives)

In [ ]:
# plot accuracy precision and recall according to different thresholds
thresholds = np.linspace(0, 1, 100)
accuracies = []
precisions = []
recalls = []
f1_scores = []

for threshold in thresholds:
    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0
    for outputs, labels in result_computed:
        predicted_class = outputs > threshold
        true_class = labels
        if true_class == 1 and predicted_class == 1:
            true_positives += 1
        elif true_class == 0 and predicted_class == 0:
            true_negatives += 1
        elif true_class == 0 and predicted_class == 1:
            false_positives += 1
        elif true_class == 1 and predicted_class == 0:
            false_negatives += 1
    accuracy = (true_positives + true_negatives) / (true_positives + true_negatives + false_positives + false_negatives)
    if true_positives > 0:
        precision = true_positives / (true_positives + false_positives)
    else:
        precision = 0
    if true_positives > 0:
        recall = true_positives / (true_positives + false_negatives)
    else:
        recall = 0
    #f1_score = 2 * precision * recall / (precision + recall)
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1_score)
    
import matplotlib.pyplot as plt
plt.figure(figsize=(20,10))
plt.plot(thresholds, accuracies, label='Accuracy')
plt.plot(thresholds, precisions, label='Precision')
plt.plot(thresholds, recalls, label='Recall')
plt.ylim([0.5,1])
plt.legend()
# set figure size



In [ ]:
os.path.exists(r"/home/varhai/PRIMUS/var_vestlontest/export/images")

In [ ]:
from torchvision.transforms import Invert


In [ ]:
train_df

In [ ]:
classifier_is_top.to("cpu")

In [ ]:
no_sutures_df

In [ ]:
class InvertGrayscalePIL:
    def __call__(self, image):
        return ImageOps.invert(image)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Lambda(InvertGrayscalePIL()), 
    transforms.Lambda(lambda x: x.crop((60, 100, x.width-60, x.height-120))),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])


In [ ]:
val_losses, epoch_losses

In [ ]:
list(test_df.index).index(3991)

In [ ]:
list(test_df[test_df['STRAIN']=='C57Bl/6NCrl'] ['Unnamed: 0'])

In [ ]:
test_df.iloc[:10]

In [ ]:
max(['Ch097' in x for x in list(test_df[test_df['STRAIN']=='C57Bl/6NCrl'] ['Unnamed: 0'])])

In [36]:
train_data

[{'points': array([[-392.86225789, -251.23279121,  516.83347705],
         [-602.16918878, -223.07576944,  503.7316459 ],
         [-488.35437767, -227.74656778,  653.28953623]]),
  'image': array([[[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],
  
         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],
  
         [[0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          ...,
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.],
          [0., 0., 0., ..., 0., 0., 0.]],
  
         ...,
  